# 04d — Comparison Arena

## The One-Sentence Version
Same data, all methods, side by side — see what each reveals and what each misses.

## What You'll Build Intuition For
- How PCA, t-SNE, and UMAP behave on the same datasets
- When nonlinear methods genuinely add value over PCA
- When PCA is sufficient and nonlinear methods are overkill

## Prerequisites
- [03c — PCA Applied](../03_linear_methods/03c_pca_applied.ipynb)
- [04b — t-SNE Explained](04b_tsne_explained.ipynb)
- [04c — UMAP Explained](04c_umap_explained.ipynb)

## The Story

We've now learned three embedding methods: PCA (linear), t-SNE (nonlinear, local focus), and UMAP (nonlinear, better global structure). Each has its strengths. But when should you use which?

The best way to build this judgement is to see them compete. We'll throw four very different datasets at all three methods and see what happens. The results will surprise you — sometimes PCA wins.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import (make_digit_data, make_swiss_roll_data,
                                   make_patient_data, make_low_d_in_high_d)

apply_style()
rng = np.random.default_rng(42)

In [ ]:
def arena_plot(X, labels, title, cmap="viridis", is_categorical=False):
    """Run PCA, t-SNE, UMAP and plot side by side."""
    # Fit all three
    X_pca = PCA(n_components=2).fit_transform(X)
    X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)
    X_umap = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        random_state=42).fit_transform(X)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    methods = [("PCA", X_pca), ("t-SNE", X_tsne), ("UMAP", X_umap)]

    for ax, (name, X_emb) in zip(axes, methods):
        if is_categorical:
            unique_labels = sorted(set(labels))
            for lab in unique_labels:
                mask = labels == lab
                ax.scatter(X_emb[mask, 0], X_emb[mask, 1], s=8, alpha=0.5,
                           color=PALETTE[int(lab) % len(PALETTE)],
                           label=str(int(lab)))
        else:
            ax.scatter(X_emb[:, 0], X_emb[:, 1], s=8, alpha=0.5,
                       c=labels, cmap=cmap)
        ax.set_title(name, fontsize=13)
        ax.set_xticks([])
        ax.set_yticks([])

    if is_categorical:
        axes[-1].legend(markerscale=3, fontsize=8, ncol=2, loc="upper right")

    fig.suptitle(title, fontweight="bold", fontsize=14)
    plt.tight_layout()
    return fig

## Swiss Roll Arena

A 2D manifold curled up in 3D. The colour encodes position along the roll — nearby colours should stay nearby.

In [ ]:
X_swiss, colour_swiss = make_swiss_roll_data(n=1500, noise=0.3, seed=42)

fig = arena_plot(X_swiss, colour_swiss, "Swiss Roll: nonlinear methods win decisively")
plt.savefig("visuals/04d_arena_swiss.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA: colours overlap — can't unroll the manifold.")
print("t-SNE: preserves local colour gradient nicely.")
print("UMAP: similar quality, often cleaner global arrangement.")
print("\nVerdict: nonlinear methods are essential here.")

## Digits Arena

64-dimensional digit images. 10 classes that should form distinct clusters.

In [ ]:
data, target, images = make_digit_data()

fig = arena_plot(data, target, "Digits: all methods find clusters, nonlinear is sharper",
                 is_categorical=True)
plt.savefig("visuals/04d_arena_digits.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA: some separation, but classes overlap.")
print("t-SNE: tight, well-separated clusters.")
print("UMAP: tight clusters with more meaningful global layout.")
print("\nVerdict: PCA gives a first look; nonlinear methods show the full picture.")

## Patient Data Arena

Synthetic patient data: 5 signal dimensions + 50 noise dimensions. No class labels — just continuous variation. Does nonlinear embedding add anything?

In [ ]:
X_patients, feature_names, signal_mask = make_patient_data(
    n=500, d_signal=5, d_noise=50, seed=42
)

# Colour by first signal feature (Age) for visual reference
colour_patients = X_patients[:, 0]  # Age

fig = arena_plot(X_patients, colour_patients,
                 "Patient Data: PCA handles this well — nonlinear adds modest value")
plt.savefig("visuals/04d_arena_patients.png", dpi=150, bbox_inches="tight")
plt.show()

print("The patient data has linear structure (correlated features + noise).")
print("PCA captures the signal-noise separation cleanly.")
print("t-SNE/UMAP may find subtle nonlinear patterns but don't dramatically improve things.")
print("\nVerdict: PCA is sufficient. Nonlinear methods add complexity without clear benefit.")

## When Linear Wins

Here's the counter-example: data that is perfectly linear (a Gaussian cloud). PCA captures everything. What do t-SNE and UMAP do?

In [ ]:
# Purely linear data: Gaussian cloud in 20D with 3 dominant directions
X_linear, Z_true = make_low_d_in_high_d(n=800, intrinsic_d=3, ambient_d=20, seed=42)
colour_linear = Z_true[:, 0]  # colour by first latent dimension

fig = arena_plot(X_linear, colour_linear,
                 "Linear Gaussian data: PCA is perfect, nonlinear methods distort")
plt.savefig("visuals/04d_arena_linear.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA: clean colour gradient — captures the linear structure perfectly.")
print("t-SNE: distorts the Gaussian into artificial clusters (a known artefact).")
print("UMAP: similar distortion — creates structure where none exists.")
print("\nVerdict: don't use a chainsaw to cut butter. PCA is the right tool here.")

> **Key insight:** t-SNE and UMAP can create apparent clusters in data that has no real cluster structure. On Gaussian data, both methods fragment the smooth distribution into artificial clumps. Always run PCA first — if it looks clean, you may not need nonlinear methods.

## Summary: When to Use What

| Dataset Type | Best Method | Why |
|-------------|-------------|-----|
| Linear structure (Gaussian, correlated features) | **PCA** | Captures the structure perfectly; nonlinear methods distort |
| Curved manifolds (Swiss roll, S-curve) | **UMAP or t-SNE** | PCA can't follow curvature |
| Discrete clusters in high-D | **UMAP** (default) or **t-SNE** | Both excel; UMAP is faster and has better global layout |
| Large datasets (>10k points) | **UMAP** | t-SNE too slow; PCA if structure is linear |
| Quick exploration | **PCA first**, then UMAP | PCA is instant and reveals if nonlinear methods are needed |

<details>
<summary><b>The Maths Behind This</b></summary>

The three methods optimise fundamentally different objectives:

**PCA:** $\min_W \sum_i \|\mathbf{x}_i - WW^T \mathbf{x}_i\|^2$ — minimise reconstruction error in a linear subspace.

**t-SNE:** $\min_Y \text{KL}(P \| Q) = \sum_{ij} p_{ij} \log \frac{p_{ij}}{q_{ij}}$ — match high-D neighbour probabilities to low-D neighbour probabilities.

**UMAP:** $\min_Y \text{CE}(W, V) = \sum_{ij} [w_{ij} \log \frac{w_{ij}}{v_{ij}} + (1-w_{ij}) \log \frac{1-w_{ij}}{1-v_{ij}}]$ — match fuzzy graph edge weights.

PCA's objective is global and linear. t-SNE's is local and asymmetric (only penalises missed neighbours). UMAP's is local but symmetric (penalises both missed and false neighbours). These different objectives explain why they behave differently on the same data.

</details>

## Where This Matters

**Exploratory data analysis:** The arena approach — running multiple methods side by side — is the gold standard for EDA. If PCA and UMAP tell the same story, you can be confident in the structure. If they disagree, that's a signal to investigate further.

**Communicating results:** PCA plots are easier to explain to stakeholders ("these two axes capture 85% of the variation"). UMAP plots are prettier and more intuitive but harder to justify formally. Choose based on your audience.

## Feynman Check

1. **For each of the four test datasets, which method would you recommend and why?**

2. **When is PCA sufficient and nonlinear methods are overkill?**

---

Module 04 is complete. You now have a toolkit for both linear and nonlinear dimensionality reduction — and the judgement to know when each is appropriate.

In **Module 05: Feature Selection**, we'll take a different approach entirely: instead of projecting data onto new axes, we'll learn to pick the *original features* that matter most.